# META-CXR Table 6 — BERTScore với MedGemma từ GCS checkpoints

Notebook reproduce **Table 6** từ paper *Chest X-Ray Report Generation Using Abnormality Guided Vision Language Model* (IEEE Access 2025, DOI 10.1109/ACCESS.2025.3606961).

**Bảng**: `BERTScore` cho radiology report generation, cho mỗi combo của 3 vision encoder **RN50 / ViT / Swin** (7 rows; paper liệt kê 6/7).

## ⚠ Khác biệt so với paper

Paper dùng **Vicuna-7B + LoRA** đã được fine-tune với projection layer `llm_proj` (768 → 4096) huấn luyện chung. Notebook này swap sang **MedGemma + LoRA của Google** (user cung cấp HF model ID).

Vì Google MedGemma LoRA **không** được train chung với `llm_proj` của META-CXR, projection 768 → 4096 không transfer được sang MedGemma. Hai lựa chọn:

- **Approach mặc định (notebook này): text-only prompt.** Không dùng soft prompt từ Q-Former.   Chỉ pass abnormality findings (P/N/U strings từ MHCAC) làm input text cho MedGemma.   Giữ tính meaningful của ablation: encoder combo khác → classification khác → prompt khác → report khác.
- *Optional alt:* thêm MedGemma vision-enabled: re-init `llm_proj` 768→3072 cho Gemma-2-2B rồi fine-tune trên MIMIC-CXR train set.   Out-of-scope cho 1 notebook eval; user implement nếu cần.

**Tiêu chí pass (relaxed vì LLM swap):** mỗi run có BERTScore F1 > 0.20. Δ tuyệt đối với paper KHÔNG có ý nghĩa vì model+prompt khác hẳn; quan tâm xu hướng *tương đối* (combo nhiều encoder hơn → BERTScore cao hơn).

## Yêu cầu trên Kaggle

1. **Kaggle Secrets**:
   - `GCS_SERVICE_ACCOUNT` — service-account JSON có quyền `roles/storage.objectViewer` trên bucket `meta-cxr-checkpoint`. Cũng hỗ trợ `GCP_SERVICE_ACCOUNT_JSON` hoặc `GCP_SERVICE_ACCOUNT_B64`.
   - `HF_TOKEN` — Hugging Face token có quyền access MedGemma model (nếu gated).
2. Attach Kaggle datasets under `/kaggle/input/datasets/phuong20052/`: `mimic-cxr-jpg-lite`, `mimic-cxr-p10-processed`. Source code `META-CXR` sẽ được clone/pull vào `/kaggle/working/META-CXR`; nếu đã attach source dataset thì setup vẫn tự nhận diện được.
3. Bật **Internet**. Cell dependency giữ `numpy`/`pandas` ở major version 2 để tương thích Kaggle/Python mới. GPU **T4 ×2** hoặc **A100**.
4. Trước khi chạy: ở **Cell 5** điền `MEDGEMMA_MODEL_ID` và `MEDGEMMA_LORA_ID`.

## Output

Bảng 7 hàng: `RN50`, `ViT`, `Swin`, `BERTScore`, `Paper BERTScore`, `Δ vs Paper`. Hàng nào không có `checkpoint_best.pth` trên GCS → `BERTScore` ghi `MISSING`. CSV lưu tại `/kaggle/working/encoder_bertscore_table.csv`.

In [1]:
"""
Cell 1 — Install dependencies (Kaggle).

Chạy cell này đầu tiên sau khi Restart Kernel. Cell này giữ numpy và
pandas ở major version 2 để tương thích với Kaggle/Python mới.
"""
import shutil
import subprocess
import sys
from pathlib import Path



def pip_install(*packages):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "--upgrade-strategy", "only-if-needed", *packages,
        ],
        check=True,
    )


PACKAGES = [
    # OpenCV 4.12 requires numpy>=2,<2.3 on Python >=3.9.
    "opencv-python>=4.12,<4.13",
    "scikit-image>=0.22",
    "scikit-learn>=1.4",
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal==0.2.2",
    "timm>=0.9.0",
    "spacy>=3.8,<3.9",
    "nltk>=3.9",
    "google-cloud-storage",
    # Gemma 3 / MedGemma checkpoints need Transformers with `gemma3` support.
    "transformers>=4.53.0,<5",
    "tokenizers>=0.21,<0.22",
    # accelerate / safetensors / torchao / peft: required by transformers 4.53+
    # for Gemma 3 / MedGemma model loading with device_map + LoRA.
    "accelerate>=0.34",
    "safetensors>=0.5",
    "torchao>=0.16.0",
    "peft>=0.14.0",
    "bert-score>=0.3.13",
    "sentencepiece",
]
pip_install(*PACKAGES)
pip_install(
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

java_bin = shutil.which("java")
JAVA_HOME_DETECTED = (
    str(Path(java_bin).resolve().parent.parent)
    if java_bin
    else "/usr/lib/jvm/java-11-openjdk-amd64"
)
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

import cv2 as _cv2
import numpy as _np
import pandas as _pd
import torch as _torch
import transformers as _transformers
from packaging.version import Version as _Version

if _Version(_transformers.__version__) < _Version("4.53.0"):
    raise RuntimeError(
        f"transformers phải >= 4.53.0 để load Gemma 3/MedGemma, "
        f"hiện tại là {_transformers.__version__}. "
        "Nếu vừa upgrade trong kernel cũ, hãy Restart Kernel/Session rồi chạy lại từ đầu."
    )

if not _np.__version__.startswith("2."):
    raise RuntimeError(f"numpy phải là major version 2, hiện tại là {_np.__version__}")
if not _pd.__version__.startswith("2."):
    raise RuntimeError(f"pandas phải là major version 2, hiện tại là {_pd.__version__}")

print(f"numpy   = {_np.__version__}")
print(f"pandas  = {_pd.__version__}")
print(f"opencv  = {_cv2.__version__}")
print(f"torch   = {_torch.__version__}")
print(f"transformers = {_transformers.__version__}")
print(f"GPUs available: {_torch.cuda.device_count()}")
for _i in range(_torch.cuda.device_count()):
    print(f"  GPU {_i}: {_torch.cuda.get_device_name(_i)}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.2 MB/s eta 0:00:00
Detected JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
numpy   = 2.0.2
pandas  = 2.3.3
opencv  = 4.12.0
torch   = 2.10.0+cu128
transformers = 4.55.4
GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}, pulling latest changes...")
    !git -C {REPO_DIR} pull

# Change working directory to repo root
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

Repository already exists at /kaggle/working/META-CXR, pulling latest changes...
Already up to date.
Working directory: /kaggle/working/META-CXR
total 408
drwxr-xr-x 14 root root  4096 May 25 07:25 .
drwxr-xr-x  4 root root  4096 May 25 07:34 ..
drwxr-xr-x  2 root root  4096 May 25 07:21 assets
drwxr-xr-x  3 root root  4096 May 25 07:21 biovil_t
-rw-r--r--  1 root root   538 May 25 07:21 build_container.sh
drwxr-xr-x  3 root root  4096 May 25 07:21 checkpoints
-rw-r--r--  1 root root  8919 May 25 07:21 CHECKPOINT_WORKFLOW.md
drwxr-xr-x  4 root root  4096 May 25 07:21 cloud
drwxr-xr-x  2 root root  4096 May 25 07:21 configs
-rw-r--r--  1 root root    90 May 25 07:21 Dockerfile
-rw-r--r--  1 root root  2612 May 25 07:21 eval_guild
-rw-r--r--  1 root root 10120 May 25 07:21 eval_paper_style.py
-rw-r--r--  1 root root  7474 May 25 07:21 generate_mimic_cxr_cleaned.ipynb
drwxr-xr-x  8 root root  4096 May 25 07:35 .git
-rw-r--r--  1 root root   247 May 25 07:21 .gitignore
-rw-r--r--  1 root r

In [3]:
import os
import sys
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')
KAGGLE_DATASETS_BASE = Path('/kaggle/input/datasets/phuong20052')

def is_meta_cxr_project(path: Path) -> bool:
    return (path / 'model' / 'lavis').exists() and (path / 'pretraining').exists()

project_candidates = [Path.cwd(), WORK_DIR / 'META-CXR', KAGGLE_DATASETS_BASE, KAGGLE_DATASETS_BASE / 'META-CXR']
if INPUT_DIR.exists():
    for root in INPUT_DIR.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])
if KAGGLE_DATASETS_BASE.exists():
    for root in KAGGLE_DATASETS_BASE.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])

source_project = next((p for p in project_candidates if is_meta_cxr_project(p)), None)
if source_project is None:
    raise FileNotFoundError('Không tìm thấy code META-CXR.')

PROJECT_DIR = WORK_DIR / 'META-CXR'
if source_project.resolve() != PROJECT_DIR.resolve():
    if not (PROJECT_DIR.exists() and is_meta_cxr_project(PROJECT_DIR)):
        ignore = shutil.ignore_patterns('.git', 'wandb', '__pycache__', '*.pyc', 'output', 'outputs')
        shutil.copytree(source_project, PROJECT_DIR, dirs_exist_ok=True, ignore=ignore)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'model'))

def first_existing(paths):
    for item in paths:
        path = Path(item)
        if path.exists():
            return path
    raise FileNotFoundError('Không tìm thấy path nào trong: ' + ', '.join(map(str, paths)))

IMAGE_ROOT = first_existing([
    KAGGLE_DATASETS_BASE / 'mimic-cxr-jpg-lite',
    '/kaggle/input/mimic-cxr-jpg-lite',
    '/kaggle/input/datasets/mimic-cxr-jpg-lite',
])
PROCESSED_ROOT = first_existing([
    KAGGLE_DATASETS_BASE / 'mimic-cxr-p10-processed',
    '/kaggle/input/mimic-cxr-p10-processed',
    '/kaggle/input/datasets/mimic-cxr-p10-processed',
])
CHECKPOINT_ROOT = Path('/kaggle/temp/checkpoints')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

java_home = globals().get('JAVA_HOME_DETECTED')
if not java_home:
    import subprocess
    result = subprocess.run(
        "readlink -f $(which java) | sed 's|/bin/java||'",
        shell=True,
        capture_output=True,
        text=True,
    )
    java_home = result.stdout.strip() or '/usr/lib/jvm/java-11-openjdk-amd64'
java_path = java_home + '/bin:'

GCS_PROJECT = os.environ.get('GCS_PROJECT', 'mimic-cxr-jpg-491409')
GCS_BUCKET = os.environ.get('GCS_BUCKET', 'gs://meta-cxr-checkpoint')

(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
(PROJECT_DIR / 'configs' / 'env_config.yaml').write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/temp/output"
  checkpoint_dir: "{CHECKPOINT_ROOT}"
  gcs_bucket: "{GCS_BUCKET}"
  gcs_project: "{GCS_PROJECT}"
wandb:
  entity: "phuongnm150505-uit"
  project: "meta-cxr-encoder-comparison"
java:
  home: "{java_home}"
  path: "{java_path}"
''')

print('PROJECT_DIR    =', PROJECT_DIR)
print('IMAGE_ROOT     =', IMAGE_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

PROJECT_DIR    = /kaggle/working/META-CXR
IMAGE_ROOT     = /kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite
PROCESSED_ROOT = /kaggle/input/datasets/phuong20052/mimic-cxr-p10-processed
CHECKPOINT_ROOT= /kaggle/temp/checkpoints


## Cell 4 — GCS auth + lazy checkpoint download

Same Google Cloud Storage client helpers as Notebook 1 (Table 5).

In [4]:
import base64
import json as _json
import os
from pathlib import Path

from google.cloud import storage
from google.oauth2 import service_account

GCS_PROJECT = os.environ.get("GCS_PROJECT", "mimic-cxr-jpg-491409")
GCS_BUCKET_NAME = os.environ.get("GCS_BUCKET", "meta-cxr-checkpoint").replace("gs://", "").rstrip("/")
GCS_BUCKET = f"gs://{GCS_BUCKET_NAME}"
CHECKPOINT_FILENAME = "checkpoint_best.pth"


def _get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(name)


    except Exception:
        return None


def _load_service_account_info():
    raw = _get_secret("GCS_SERVICE_ACCOUNT") or _get_secret("GCP_SERVICE_ACCOUNT_JSON") or _get_secret("GCP_SERVICE_ACCOUNT_B64")
    if not raw:
        return None

    raw = raw.strip()
    try:
        return _json.loads(raw)
    except _json.JSONDecodeError:
        return _json.loads(base64.b64decode(raw).decode("utf-8"))


def build_storage_client(required=True):
    info = _load_service_account_info()
    if info:
        credentials = service_account.Credentials.from_service_account_info(info)
        return storage.Client(project=GCS_PROJECT, credentials=credentials)

    adc_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if adc_path and os.path.exists(adc_path):
        return storage.Client(project=GCS_PROJECT)

    if required:
        raise RuntimeError(
            "GCS credentials not found. Add Kaggle Secret GCS_SERVICE_ACCOUNT "
            "with a service-account JSON key or base64-encoded JSON. "
            "Also accepted: GCP_SERVICE_ACCOUNT_JSON or GCP_SERVICE_ACCOUNT_B64. "
            f"The service account needs read access to {GCS_BUCKET}."
        )
    return None


def find_gcs_checkpoint_blob(client, run: str, filename: str = CHECKPOINT_FILENAME):
    candidates = []
    for blob in client.list_blobs(GCS_BUCKET_NAME, prefix=f"{run}/"):
        if blob.name == f"{run}/{filename}" or blob.name.endswith(f"/{filename}"):
            candidates.append(blob)
    if not candidates:
        return None
    return sorted(candidates, key=lambda b: ((b.updated.timestamp() if b.updated else 0), b.name))[-1]


storage_client = build_storage_client(required=True)
print(f"GCS bucket: {GCS_BUCKET}")
print(f"GCS project: {GCS_PROJECT}")


def ensure_local_checkpoint(run: str):
    """Ensure CHECKPOINT_ROOT/run/checkpoint_best.pth exists locally, downloading from GCS if needed."""
    local_dir = CHECKPOINT_ROOT / run
    local_dir.mkdir(parents=True, exist_ok=True)
    local = local_dir / CHECKPOINT_FILENAME
    if local.exists():
        return local

    blob = find_gcs_checkpoint_blob(storage_client, run, CHECKPOINT_FILENAME)
    if blob is None:
        return None

    blob.download_to_filename(str(local))
    print(f"Downloaded checkpoint: {GCS_BUCKET}/{blob.name} -> {local}")
    return local


GCS bucket: gs://meta-cxr-checkpoint
GCS project: mimic-cxr-jpg-491409


## Cell 5 — MedGemma config

**Bắt buộc điền** `MEDGEMMA_MODEL_ID` và (tùy chọn) `MEDGEMMA_LORA_ID` trước khi chạy notebook.

Notebook này load MedGemma/Gemma 3 multimodal bằng `AutoModelForImageTextToText` + `AutoProcessor`.
Một số lựa chọn phù hợp:

- `google/medgemma-1.5-4b-it` — multimodal, nhẹ hơn cho Kaggle T4.
- `google/medgemma-4b-it` — multimodal.
- `google/medgemma-27b-it` — multimodal lớn hơn, cần GPU mạnh hơn.

Notebook sẽ raise `ValueError` nếu placeholder `<FILL_IN>` chưa được thay.

In [5]:
import os

# === BẮT BUỘC ĐIỀN TRƯỚC KHI CHẠY ===
MEDGEMMA_MODEL_ID: str = 'google/medgemma-1.5-4b-it'   # ví dụ: 'google/medgemma-4b-it' hoặc 'google/medgemma-27b-it'
MEDGEMMA_LORA_ID:  str = 'DeepRadiology/medgemma1.5-CXR'            # ví dụ: 'google/your-medical-lora'; để '' nếu không dùng LoRA

# Sinh-text params
MAX_NEW_TOKENS = 256
NUM_BEAMS      = 1
SEED           = 16

# Inference batching
EVAL_BATCH_SIZE = 1
NUM_WORKERS     = 2

# Mặc định: chỉ chạy trên subset test (để demo nhanh). Đặt None để chạy full test.
TEST_SAMPLE_LIMIT = 200  # set to None for full test set

if '<FILL_IN>' in MEDGEMMA_MODEL_ID:
    raise ValueError('Hãy đặt MEDGEMMA_MODEL_ID ở cell này trước khi chạy.')

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
except Exception:
    pass

print('MEDGEMMA_MODEL_ID =', MEDGEMMA_MODEL_ID)
print('MEDGEMMA_LORA_ID  =', MEDGEMMA_LORA_ID or '(none)')
print('TEST_SAMPLE_LIMIT =', TEST_SAMPLE_LIMIT or 'full test set')

MEDGEMMA_MODEL_ID = google/medgemma-1.5-4b-it
MEDGEMMA_LORA_ID  = DeepRadiology/medgemma1.5-CXR
TEST_SAMPLE_LIMIT = 200


In [6]:
import gc
import json
import os
import random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import transformers as hf_transformers
from packaging.version import Version
from transformers import AutoProcessor, AutoModelForImageTextToText

# Bật synchronous CUDA launch để trace chính xác dòng gây lỗi.
# Có thể tắt (comment) sau khi notebook chạy ổn định để tăng perf.
os.environ.setdefault('CUDA_LAUNCH_BLOCKING', '1')

MIN_TRANSFORMERS_VERSION = "4.53.0"
if Version(hf_transformers.__version__) < Version(MIN_TRANSFORMERS_VERSION):
    raise RuntimeError(
        f"transformers>={MIN_TRANSFORMERS_VERSION} is required for Gemma 3/MedGemma; "
        f"current version is {hf_transformers.__version__}. "
        "Run Cell 1, restart the kernel/session, then run the notebook again."
    )

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler
from model.lavis.datasets.builders import *
from model.lavis.models import *
from model.lavis.processors import *
from model.lavis.tasks import *
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

registry.mapping['paths']['cache_root'] = '.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Seeding cho reproducibility.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

ABNORMALITIES_14 = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
]
CLASS_MAP = {'negative': 0, 'positive': 1, 'uncertain': 2}

with open(PROJECT_DIR / 'threshold.json') as f:
    THRESHOLDS = json.load(f)

print('DEVICE =', DEVICE)
print('transformers =', hf_transformers.__version__)


2026-05-25 07:36:11.525976: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779694571.551960    1843 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779694571.559565    1843 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779694571.596831    1843 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779694571.596848    1843 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779694571.596851    1843 computation_placer.cc:177] computation placer alr

DEVICE = cuda
transformers = 4.55.4


/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [7]:
TABLE_RUNS = [
    {'run': '01_biovil_only',        'RN50': True,  'ViT': False, 'Swin': False},
    {'run': '02_pubmedclip_only',    'RN50': False, 'ViT': True,  'Swin': False},
    {'run': '03_swin_only',          'RN50': False, 'ViT': False, 'Swin': True},
    {'run': '04_biovil_pubmedclip',  'RN50': True,  'ViT': True,  'Swin': False},
    {'run': '05_biovil_swin',        'RN50': True,  'ViT': False, 'Swin': True},
    {'run': '06_pubmedclip_swin',    'RN50': False, 'ViT': True,  'Swin': True},
    {'run': '07_all_three',          'RN50': True,  'ViT': True,  'Swin': True},
]

# Paper Table 6 BERTScore (Vicuna-7B + LoRA pipeline). Notebook này dùng MedGemma,
# nên Δ tuyệt đối chỉ tham khảo; xu hướng tương đối giữa các combo mới có ý nghĩa.
PAPER_BERTSCORE = {
    '01_biovil_only':       0.312,
    '02_pubmedclip_only':   0.289,
    '03_swin_only':         0.267,
    '04_biovil_pubmedclip': 0.401,
    '05_biovil_swin':       0.394,
    '06_pubmedclip_swin':   None,
    '07_all_three':         0.426,
}

In [8]:
def build_cfg(run_name: str):
    cfg_path = PROJECT_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_name}.yaml'
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)

def load_torch_checkpoint(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def filter_state_dict_for_model(model, state_dict):
    """Drop checkpoint tensors whose shapes do not match the current code build.

    `strict=False` ignores missing/unexpected keys, but PyTorch still raises on
    same-name shape mismatches. This handles tokenizer-vocab drift such as
    Qformer.cls.predictions.bias: 30523 in checkpoint vs 30522 in current model.
    """
    model_state = model.state_dict()
    filtered = {}
    mismatched = []
    for key, value in state_dict.items():
        if key in model_state and hasattr(value, 'shape'):
            ckpt_shape = tuple(value.shape)
            model_shape = tuple(model_state[key].shape)
            if ckpt_shape != model_shape:
                mismatched.append((key, ckpt_shape, model_shape))
                continue
        filtered[key] = value
    return filtered, mismatched


def build_meta_cxr_model(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = load_torch_checkpoint(checkpoint_path)
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    state_dict, mismatched = filter_state_dict_for_model(model, state_dict)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(
        f'{run_name}: loaded {checkpoint_path.name}; '
        f'missing={len(missing)}, unexpected={len(unexpected)}, mismatched_skipped={len(mismatched)}'
    )
    if mismatched:
        for key, ckpt_shape, model_shape in mismatched[:8]:
            print(f'  skipped shape mismatch: {key}: checkpoint={ckpt_shape}, model={model_shape}')
        if len(mismatched) > 8:
            print(f'  ... skipped {len(mismatched) - 8} more mismatched tensors')

    # Move to target device, handling any meta tensors left from model init.
    # `to_empty` creates uninitialized CPU memory that must be zeroed before
    # GPU transfer to avoid NaN/Inf triggering CUDA device-side asserts.
    try:
        model.to(DEVICE)
    except NotImplementedError:
        model.to_empty(device=torch.device('cpu'))
        for p in model.parameters():
            p.data.zero_()
        model.load_state_dict(state_dict, strict=False)
        model.to(DEVICE)

    model.eval()
    return cfg, model

def make_test_loader(cfg):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split='test',
        cfg=cfg,
        truncate=TEST_SAMPLE_LIMIT,
    )
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

@torch.no_grad()
def predict_classifications(model, batch):
    """Returns cls_logits shape (B, 14, 3) on CPU."""
    image = batch['image'].to(DEVICE, non_blocking=True)
    cls_logits, _ = model.forward_image(image)
    return cls_logits.float().cpu()

def classify_with_thresholds(logits):
    """Apply thresholds (mirror inference.py:classify_abnormalities) on 14 classes of 1 image."""
    assert logits.shape == (14, 3), f'expected (14, 3), got {tuple(logits.shape)}'
    probs = torch.softmax(logits, dim=-1).tolist()
    out = {'positive': [], 'negative': [], 'uncertain': []}
    for abn, p in zip(ABNORMALITIES_14, probs):
        if abn == 'No Finding':
            continue
        thresholds_abn = THRESHOLDS.get(abn, {})
        best_cls, best_score = None, 0.0
        for cls_name, cls_idx in CLASS_MAP.items():
            threshold = thresholds_abn.get(cls_name, 0.5)
            prob = p[cls_idx]
            if prob >= threshold and prob > best_score:
                best_cls = cls_name
                best_score = prob
        if best_cls is not None:
            out[best_cls].append(abn)
    return out

def build_prompt(classifications):
    """Text-only prompt per paper Section III.E.3 (skip <image_queries>)."""
    pos = ', '.join(classifications['positive']) or 'none'
    neg = ', '.join(classifications['negative']) or 'none'
    unc = ', '.join(classifications['uncertain']) or 'none'
    return (
        f'Positive Abnormalities: {pos}. '
        f'Negative Abnormalities: {neg}. '
        f'Uncertain Abnormalities: {unc}. '
        f'Act as an expert radiologist. Using only the structured abnormality information above, '
        f'write the Findings section of a chest X-ray report. '
        f'Use a single fluent paragraph in formal radiological style. '
        f'Do not invent findings; only describe abnormalities explicitly provided. '
        f'Return only the generated findings text.'
    )

In [9]:
try:
    from peft import PeftModel
    _HAS_PEFT = True
except ImportError:
    _HAS_PEFT = False

_LLM_CACHE = {}

def _tokenizer_from_processor(processor):
    tokenizer = getattr(processor, 'tokenizer', processor)
    if tokenizer is None:
        raise RuntimeError('Không lấy được tokenizer từ AutoProcessor.')
    return tokenizer

def _preferred_llm_dtype():
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32

def _has_meta_tensors(model):
    for tensor in list(model.parameters()) + list(model.buffers()):
        if getattr(tensor, 'is_meta', False):
            return True
    return False

def _clear_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _hf_kwargs(hf_token):
    return {'token': hf_token} if hf_token else {}

def _model_input_device(model):
    hf_device_map = getattr(model, 'hf_device_map', None)
    if isinstance(hf_device_map, dict):
        for device in hf_device_map.values():
            if isinstance(device, int):
                return torch.device(f'cuda:{device}')
            if isinstance(device, str) and device not in {'cpu', 'disk', 'meta'}:
                return torch.device(device)

    device = getattr(model, 'device', None)
    if device is not None and str(device) != 'meta':
        return device

    for tensor in list(model.parameters()) + list(model.buffers()):
        if not getattr(tensor, 'is_meta', False):
            return tensor.device
    return torch.device(DEVICE)

def _load_base_medgemma(hf_token, dtype, use_device_map=True):
    kwargs = _hf_kwargs(hf_token)
    kwargs['torch_dtype'] = dtype
    if use_device_map:
        kwargs['device_map'] = 'auto'
    else:
        kwargs['low_cpu_mem_usage'] = False

    return AutoModelForImageTextToText.from_pretrained(MEDGEMMA_MODEL_ID, **kwargs)

def _activate_adapter(model, adapter_name):
    if not adapter_name:
        return
    if hasattr(model, 'set_adapter'):
        model.set_adapter(adapter_name)
    elif hasattr(model, 'active_adapters'):
        model.active_adapters = adapter_name

def _attach_medgemma_lora(model, hf_token, dtype):
    if not MEDGEMMA_LORA_ID:
        return model

    print(f'>>> Attaching LoRA {MEDGEMMA_LORA_ID} ...')

    # Transformers PEFT integration handles adapter placement better for
    # device_map-dispatched VLMs than wrapping the whole model in PeftModel.
    if hasattr(model, 'load_adapter'):
        try:
            adapter_name = model.load_adapter(MEDGEMMA_LORA_ID, **_hf_kwargs(hf_token))
        except TypeError:
            adapter_name = model.load_adapter(MEDGEMMA_LORA_ID)
        _activate_adapter(model, adapter_name)
        return model

    if not _HAS_PEFT:
        raise ImportError('peft chưa cài; chạy lại cell pip install.')

    kwargs = _hf_kwargs(hf_token)
    kwargs['torch_dtype'] = dtype
    return PeftModel.from_pretrained(model, MEDGEMMA_LORA_ID, **kwargs)

def _load_medgemma_with_retry(hf_token, dtype):
    last_exc = None

    for use_device_map in (True, False):
        mode = "device_map='auto'" if use_device_map else 'single-device fallback'
        try:
            print(f'>>> Loading {MEDGEMMA_MODEL_ID} with AutoModelForImageTextToText ({mode}, dtype={dtype}) ...')
            llm = _load_base_medgemma(hf_token, dtype, use_device_map=use_device_map)
            llm = _attach_medgemma_lora(llm, hf_token, dtype)

            if not use_device_map and DEVICE == 'cuda':
                llm = llm.to(torch.device('cuda'))

            if _has_meta_tensors(llm):
                raise NotImplementedError(
                    'MedGemma vẫn còn meta tensor sau khi load; retry bằng single-device fallback.'
                )
            return llm

        except NotImplementedError as exc:
            last_exc = exc
            if 'meta tensor' not in str(exc) and 'meta tensors' not in str(exc):
                raise
            print(f'WARN: {exc}')
            print('>>> Reloading MedGemma without device_map to materialize weights before moving to GPU ...')
            try:
                del llm
            except UnboundLocalError:
                pass
            _clear_cuda_cache()

    raise last_exc

def get_medgemma():
    """Load (model, processor) once, cache across all 7 runs."""
    if 'model' in _LLM_CACHE:
        return _LLM_CACHE['model'], _LLM_CACHE['processor']

    hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    dtype = _preferred_llm_dtype()

    processor = AutoProcessor.from_pretrained(
        MEDGEMMA_MODEL_ID,
        **_hf_kwargs(hf_token),
    )
    tokenizer = _tokenizer_from_processor(processor)

    llm = _load_medgemma_with_retry(hf_token, dtype)
    llm.eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if getattr(llm, 'generation_config', None) is not None:
        llm.generation_config.pad_token_id = tokenizer.pad_token_id

    _LLM_CACHE['model'] = llm
    _LLM_CACHE['processor'] = processor
    return llm, processor

@torch.no_grad()
def generate_report(prompt, llm, processor):
    tokenizer = _tokenizer_from_processor(processor)
    messages = [
        {
            'role': 'user',
            'content': [{'type': 'text', 'text': prompt}],
        }
    ]

    if hasattr(processor, 'apply_chat_template'):
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors='pt',
            truncation=True,
            max_length=2048,
        )
    else:
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)

    inputs = inputs.to(_model_input_device(llm))
    input_len = inputs['input_ids'].shape[1]
    out = llm.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    return text.strip()


In [10]:
from bert_score import score as bert_score_fn

BERTSCORE_MODEL = 'microsoft/deberta-xlarge-mnli'

def compute_bertscore(predictions, references):
    """Mean BERTScore F1 trên cặp (pred, ref)."""
    if not predictions:
        return float('nan')
    P, R, F1 = bert_score_fn(
        predictions,
        references,
        lang='en',
        model_type=BERTSCORE_MODEL,
        rescale_with_baseline=False,
        verbose=False,
        device='cpu',
    )
    return float(F1.mean().item())

In [11]:
def bertscore_for_run(run_name):
    """Pipeline đầy đủ cho 1 encoder-combo run.

    Returns: (mean_bertscore, num_samples_evaluated)
    """
    ckpt = ensure_local_checkpoint(run_name)
    if ckpt is None:
        raise FileNotFoundError(
            f'{run_name}: chưa có checkpoint_best.pth trên GCS '
            f'({GCS_BUCKET}/{run_name}/checkpoint_best.pth).'
        )

    cfg, model = build_meta_cxr_model(run_name, ckpt)
    loader = make_test_loader(cfg)
    llm, processor = get_medgemma()

    predictions = []
    references = []

    for batch in tqdm(loader, desc=f'{run_name} infer'):
        try:
            cls_logits = predict_classifications(model, batch)
        except RuntimeError as exc:
            if 'device-side assert' in str(exc) or 'CUDA error' in str(exc):
                print(f'  CUDA assert during classification for batch around sample {len(predictions)}: {exc}')
                print(f'  Skipping batch and continuing...')
                for i in range(len(batch.get('image', []))):
                    predictions.append('')
                    ref_field = batch.get('text_output')
                    ref = ref_field[i] if isinstance(ref_field, (list, tuple)) else str(ref_field[i]) if ref_field is not None else ''
                    references.append(ref)
                continue
            raise

        for i in range(cls_logits.shape[0]):
            logits_i = cls_logits[i]
            if torch.isnan(logits_i).any() or torch.isinf(logits_i).any():
                print(f'  NaN/Inf logits at sample {len(predictions)}, using empty report')
                pred = ''
            else:
                classifications = classify_with_thresholds(logits_i)
                prompt = build_prompt(classifications)
                try:
                    pred = generate_report(prompt, llm, processor)
                except Exception as exc:
                    print(f'  generate failed for sample {len(predictions)}: {exc}')
                    pred = ''
            ref_field = batch['text_output']
            ref = ref_field[i] if isinstance(ref_field, (list, tuple)) else str(ref_field[i])
            predictions.append(pred)
            references.append(ref)

    # Free META-CXR model RAM; LLM cached cho run sau.
    del model, loader
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    mean_f1 = compute_bertscore(predictions, references)

    # Lưu raw predictions để debug.
    out_jsonl = WORK_DIR / f'reports_{run_name}.jsonl'
    with open(out_jsonl, 'w') as f:
        for p, r in zip(predictions, references):
            f.write(json.dumps({'pred': p, 'ref': r}) + '\n')
    print(f'>>> Wrote {out_jsonl} ({len(predictions)} samples, mean BERTScore F1={mean_f1:.4f})')

    return mean_f1, len(predictions)

In [12]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
login(user_secrets.get_secret("HF_TOKEN"));

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
rows = []
skipped = []

for item in TABLE_RUNS:
    run_name = item['run']
    paper_value = PAPER_BERTSCORE[run_name]
    try:
        bs, n = bertscore_for_run(run_name)
        rows.append({
            'RN50': '✓' if item['RN50'] else '–',
            'ViT': '✓' if item['ViT'] else '–',
            'Swin': '✓' if item['Swin'] else '–',
            'BERTScore': round(bs, 4),
            'N samples': n,
            'Paper BERTScore': paper_value if paper_value is not None else '—',
            'Delta vs Paper': round(bs - paper_value, 4) if paper_value is not None else '—',
        })
    except FileNotFoundError as exc:
        print(f'WARNING: skipping {run_name}: {exc}')
        skipped.append(run_name)
        rows.append({
            'RN50': '✓' if item['RN50'] else '–',
            'ViT': '✓' if item['ViT'] else '–',
            'Swin': '✓' if item['Swin'] else '–',
            'BERTScore': 'MISSING',
            'N samples': 0,
            'Paper BERTScore': paper_value if paper_value is not None else '—',
            'Delta vs Paper': '—',
        })

table = pd.DataFrame(
    rows,
    columns=['RN50', 'ViT', 'Swin', 'BERTScore', 'N samples', 'Paper BERTScore', 'Delta vs Paper'],
)
table.to_csv('/kaggle/working/encoder_bertscore_table.csv', index=False)

print()
print('Skipped runs (no checkpoint_best.pth on GCS):', skipped or 'none')
print(f'LLM: {MEDGEMMA_MODEL_ID}; LoRA: {MEDGEMMA_LORA_ID or "(none)"}')
print(f'Sample limit: {TEST_SAMPLE_LIMIT or "full test set"}')
print()

display(
    table.style
    .hide(axis='index')
    .set_caption('Report Generation BERTScore (MedGemma replacement for Vicuna)')
)

table

BertLMHeadModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.out

01_biovil_only: loaded checkpoint_best.pth; missing=211, unexpected=0, mismatched_skipped=1
  skipped shape mismatch: Qformer.cls.predictions.bias: checkpoint=(30523,), model=(30522,)


/kaggle/working/META-CXR/model/lavis/data/ReportDataset.py:261: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.chexpert['No Finding'].fillna(0.0, inplace=True)
/kaggle/working/META-CXR/model/lavis/data/ReportDataset.py:258: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

Number of chexpert records: 227827
Number of annotation records: 200
Number of annotation records: 200
setting up scorers...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

  generate failed for sample 0: Skip calling `torch.compiler.disable()`d function
  Explanation: Skip calling function `<function AlignDevicesHook.pre_forward at 0x794add77be20>` since it was wrapped with `torch.compiler.disable` (reason: None)
  Hint: Remove the `torch.compiler.disable` call

  Developer debug context: <function AlignDevicesHook.pre_forward at 0x794add77be20>

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0098.html

from user code:
   File "/usr/local/lib/python3.12/dist-packages/accelerate/hooks.py", line 187, in new_forward
    args, kwargs = module._hf_hook.pre_forward(module, *args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/hooks.py", line 50, in wrapper
    return wrapper._compiled_fn(*args, **kwargs)

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOG

>>> Wrote /kaggle/working/reports_01_biovil_only.jsonl (200 samples, mean BERTScore F1=0.0000)


BertLMHeadModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.out

02_pubmedclip_only: loaded checkpoint_best.pth; missing=1, unexpected=0, mismatched_skipped=1
  skipped shape mismatch: Qformer.cls.predictions.bias: checkpoint=(30523,), model=(30522,)
Number of chexpert records: 227827
Number of annotation records: 200
Number of annotation records: 200
setting up scorers...


02_pubmedclip_only infer:   0%|          | 0/200 [00:00<?, ?it/s]